# TOM: XTTS v2 Türkçe fine-tune (Colab, A100)

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **A100 GPU**.

1. Yerelde veri setini hazırla ve zip'le:
   `PYTHONPATH=. .venv/bin/python -m tools.finetune.build_dataset podcastlar/ --out data/xtts_dataset --zip`
2. `data/xtts_dataset.zip` dosyasını Drive'da **`MyDrive/TOM_AI/xtts_dataset.zip`** yoluna yükle.
3. Hücreleri sırayla çalıştır ("Tümünü çalıştır" da olur; oturum yeniden başlamaz).

Checkpoint'ler Drive'a yazılır: bağlantı koparsa `CONTINUE_PATH` ile kaldığın yerden sürdürebilirsin.
Sürümler yereldeki TOM ortamıyla aynıdır (Python 3.10, torch 2.6 cu124, coqui-tts 0.25.3, transformers 4.46.2);
farklı sürümle eğitilen checkpoint yerelde açılmayabilir.

## 1. Kurulum (~3–5 dk)
Colab'ın Python'u (3.12+) coqui-tts 0.25.3'ü desteklemiyor. Bu yüzden `uv` ile ayrı bir **Python 3.10**
ortamı (`/content/venv`) kurulur; eğitim ve değerlendirme o ortamda çalışır. Colab'ın kendi Python'una
dokunulmaz, oturum yeniden başlamaz. Hücre tekrar çalıştırılırsa kurulumu atlar.

In [ ]:
import os
PY = '/content/venv/bin/python'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if not os.path.exists(PY):
    !pip -q install uv
    !uv venv /content/venv --python 3.10 -q
    !uv pip install -q --python {PY} torch==2.6.0 torchaudio==2.6.0 coqui-tts==0.25.3 transformers==4.46.2 "numpy<2" faster-whisper==1.2.1 speechbrain==1.0.3
!{PY} -c "import torch, transformers, TTS; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '| transformers', transformers.__version__, '| coqui-tts', TTS.__version__)"

## 2. Ayarlar ve Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VERI = 'xtts_test2'   # Drive'daki zip'in adı (uzantısız): xtts_test2 ya da xtts_test1
DATASET_ZIP = f'/content/drive/MyDrive/TOM_AI/{VERI}.zip'
OUT_DIR = f'/content/drive/MyDrive/TOM_AI/xtts_runs_{VERI}'   # checkpoint ve hazır model buraya
DATASET_DIR = f'/content/dataset_{VERI}'

EPOCHS = 15        # 30–60 dk veri için 10–20; ~10 dk veride 25. En iyi checkpoint zaten saklanır
BATCH = 8          # A100 40 GB: 8, 80 GB: 16
GRAD_ACCUM = 1
LR = 5e-6
CONTINUE_PATH = '' # yarıda kalan eğitim: OUT_DIR/egitim/tom_xtts_tr-... klasörü
assert os.path.exists(DATASET_ZIP), f'{DATASET_ZIP} yok: zip\'i Drive\'da MyDrive/TOM_AI/ altına yükle'

## 3. Veri setini aç ve kontrol et

In [ ]:
import os, zipfile, json
os.makedirs(DATASET_DIR, exist_ok=True)
zipfile.ZipFile(DATASET_ZIP).extractall(DATASET_DIR)
print(open(f'{DATASET_DIR}/rapor.json').read() if os.path.exists(f'{DATASET_DIR}/rapor.json') else '')
!head -4 {DATASET_DIR}/metadata_train.csv

## 4. Betikler

In [ ]:
%%writefile /content/train_xtts.py
"""XTTS v2 GPT fine-tune (Colab A100 için; yerelde --smoke ile duman testi).

    python train_xtts.py --dataset /content/dataset --out /content/drive/MyDrive/TOM_AI/xtts_runs
    python train_xtts.py --export-only RUN_DIR/best_model.pth --out .../xtts_runs   # yalnız dışa aktar

Veri seti: build_dataset.py çıktısı (wavs/ + metadata_train.csv + metadata_eval.csv, coqui formatı).
Yalnız GPT kısmı eğitilir; DVAE ve HiFiGAN vokoder donuk kalır (standart XTTS fine-tune).
Sürümler yereldeki çıkarım ortamıyla aynı olmalı: coqui-tts 0.25.3, transformers 4.46.2.

Çıktı: <out>/hazir_model/ → model.pth, config.json, vocab.json, speakers_xtts.pth, reference.wav
TOM'da kullanım: python -m tom --xtts <hazir_model> --ref <hazir_model>/reference.wav
"""

from __future__ import annotations

import argparse
import csv
import glob
import json
import os
import shutil
from pathlib import Path

BASE_FILES = ["model.pth", "config.json", "vocab.json", "dvae.pth", "mel_stats.pth", "speakers_xtts.pth"]


def ensure_base(base_dir: Path, revision: str = "main") -> Path:
    """Temel XTTS v2 dosyaları yoksa Hugging Face'ten (coqui/XTTS-v2) indirir."""
    base_dir.mkdir(parents=True, exist_ok=True)
    missing = [f for f in BASE_FILES if not (base_dir / f).exists()]
    if missing:
        from huggingface_hub import hf_hub_download

        for f in missing:
            print(f"İndiriliyor: {f}", flush=True)
            p = hf_hub_download("coqui/XTTS-v2", f, revision=revision)
            shutil.copy(p, base_dir / f)
    return base_dir


def dataset_summary(ds: Path) -> dict:
    import soundfile as sf

    out = {}
    for name in ("metadata_train.csv", "metadata_eval.csv"):
        with open(ds / name, encoding="utf-8") as f:
            rows = list(csv.DictReader(f, delimiter="|"))
        dur = sum(sf.info(ds / r["audio_file"]).duration for r in rows)
        out[name] = {"klip": len(rows), "dakika": round(dur / 60, 1)}
    return out


def pick_reference(ds: Path, max_s: float = 11.0) -> Path:
    """Doğrulama setindeki en uzun (≤ max_s) klip: TOM'da klon referansı olarak kullanılır."""
    import soundfile as sf

    with open(ds / "metadata_eval.csv", encoding="utf-8") as f:
        rows = list(csv.DictReader(f, delimiter="|"))
    rows += list(csv.DictReader(open(ds / "metadata_train.csv", encoding="utf-8"), delimiter="|"))
    best = max((r for r in rows if sf.info(ds / r["audio_file"]).duration <= max_s),
               key=lambda r: sf.info(ds / r["audio_file"]).duration)
    return ds / best["audio_file"]


def build(args, base: Path):
    from TTS.config.shared_configs import BaseDatasetConfig
    from TTS.tts.datasets import load_tts_samples
    from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
    from TTS.tts.models.xtts import XttsAudioConfig  # 0.25'te buraya taşındı (0.24'te gpt_trainer'daydı)

    ds = Path(args.dataset)
    dataset_cfg = BaseDatasetConfig(formatter="coqui", dataset_name="tom", path=str(ds),
                                    meta_file_train="metadata_train.csv", meta_file_val="metadata_eval.csv",
                                    language="tr")
    model_args = GPTArgs(
        max_conditioning_length=132300,  # 6 sn
        min_conditioning_length=66150,   # 3 sn
        debug_loading_failures=False,
        max_wav_length=int(args.max_audio_s * 22050),
        max_text_length=200,
        mel_norm_file=str(base / "mel_stats.pth"),
        dvae_checkpoint=str(base / "dvae.pth"),
        xtts_checkpoint=str(base / "model.pth"),
        tokenizer_file=str(base / "vocab.json"),
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )
    config = GPTTrainerConfig(
        epochs=args.epochs,
        output_path=str(Path(args.out) / "egitim"),
        model_args=model_args,
        run_name="tom_xtts_tr",
        project_name="TOM_AI",
        dashboard_logger="tensorboard",
        audio=XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000),
        batch_size=args.batch,
        batch_group_size=48,
        eval_batch_size=args.batch,
        num_loader_workers=args.workers,
        num_eval_loader_workers=min(2, args.workers),
        eval_split_max_size=256,
        print_step=25,
        plot_step=100,
        log_model_step=1000,
        save_step=args.save_step,
        save_n_checkpoints=1,
        save_checkpoints=True,
        save_best_after=0,
        print_eval=True,
        optimizer="AdamW",
        optimizer_wd_only_on_weights=True,
        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
        lr=args.lr,
        lr_scheduler="MultiStepLR",
        lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
        test_sentences=[],
        mixed_precision=False,  # XTTS GPT'de fp16 kararsız; A100'de TF32 zaten açık
    )
    model = GPTTrainer.init_from_config(config)
    train_samples, eval_samples = load_tts_samples([dataset_cfg], eval_split=True,
                                                   eval_split_max_size=config.eval_split_max_size,
                                                   eval_split_size=config.eval_split_size)
    return config, model, train_samples, eval_samples


def smoke(config, model, train_samples) -> Path:
    """Tek batch ileri + geri geçiş (optimizer yok): veri biçimi, yapılandırma ve kayıp hesabı doğru mu?"""
    import torch

    model.cuda().train()
    loader = model.get_data_loader(config, {}, is_eval=False, samples=train_samples, verbose=True, num_gpus=0)
    batch = model.format_batch(next(iter(loader)))
    batch = {k: v.cuda() if torch.is_tensor(v) else v for k, v in batch.items()}
    batch = model.format_batch_on_device(batch)
    _, loss = model.train_step(batch, None)
    loss["loss"].backward()
    grads = sum(p.grad is not None for p in model.parameters())
    print(f"DUMAN TESTİ: kayıp={loss['loss'].item():.3f} (metin {loss['loss_text_ce'].item():.3f}, "
          f"ses {loss['loss_mel_ce'].item():.3f}); gradyanı olan parametre: {grads}", flush=True)
    model.zero_grad(set_to_none=True)
    ck = Path(args_global.out) / "duman_checkpoint.pth"
    ck.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict()}, ck)  # Trainer'ın kaydettiği biçimin aynısı ("xtts." önekli)
    return ck


def export(ckpt: Path, base: Path, out: Path, reference: Path | None) -> Path:
    """Trainer checkpoint'inden çıkarıma hazır klasör: optimizer ve DVAE atılır (~1,8 GB kalır)."""
    import torch

    dst = out / "hazir_model"
    dst.mkdir(parents=True, exist_ok=True)
    state = torch.load(ckpt, map_location="cpu", weights_only=False)["model"]
    # GPTTrainer: çıkarım modeli "xtts.*" altında; "dvae.*" ve "torch_mel_spectrogram_*" yalnız eğitimde gerekir
    if any(k.startswith("xtts.") for k in state):
        state = {k: v for k, v in state.items() if k.startswith("xtts.")}
    torch.save({"model": state}, dst / "model.pth")
    for f in ("config.json", "vocab.json", "speakers_xtts.pth"):
        shutil.copy(base / f, dst / f)
    if reference:
        shutil.copy(reference, dst / "reference.wav")
    (dst / "kaynak.json").write_text(json.dumps({"checkpoint": str(ckpt)}, ensure_ascii=False, indent=2))
    print(f"Hazır model: {dst}", flush=True)
    return dst


def find_best(run_root: Path) -> Path:
    cands = sorted(glob.glob(str(run_root / "**" / "best_model*.pth"), recursive=True), key=os.path.getmtime)
    if not cands:
        cands = sorted(glob.glob(str(run_root / "**" / "checkpoint_*.pth"), recursive=True), key=os.path.getmtime)
    if not cands:
        raise SystemExit(f"{run_root} altında checkpoint yok.")
    return Path(cands[-1])


def main():
    global args_global
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", help="build_dataset.py çıktı klasörü")
    ap.add_argument("--out", required=True, help="Çıktı (Colab'da Drive'da olsun: bağlantı koparsa kaybolmaz)")
    ap.add_argument("--base-dir", default="/content/xtts_base", help="Temel XTTS v2 dosyaları (yoksa iner)")
    ap.add_argument("--base-revision", default="main")
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--batch", type=int, default=8, help="A100 40 GB için 8; 80 GB'ta 16 denenebilir")
    ap.add_argument("--grad-accum", type=int, default=1)
    ap.add_argument("--lr", type=float, default=5e-6)
    ap.add_argument("--max-audio-s", type=float, default=11.6)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--save-step", type=int, default=10000, help="Ara checkpoint sıklığı (en iyi model her epoch)")
    ap.add_argument("--continue-path", help="Yarıda kalan eğitimin klasörü (kaldığı yerden sürdür)")
    ap.add_argument("--export-only", help="Yalnız bu checkpoint'i dışa aktar")
    ap.add_argument("--smoke", action="store_true", help="Tek batch ileri/geri geçiş + dışa aktarım (eğitim yok)")
    args = args_global = ap.parse_args()

    base = ensure_base(Path(args.base_dir), args.base_revision)
    out = Path(args.out)
    if args.export_only:
        ref = pick_reference(Path(args.dataset)) if args.dataset else None
        export(Path(args.export_only), base, out, ref)
        return
    if not args.dataset:
        raise SystemExit("--dataset gerekli")
    print(json.dumps(dataset_summary(Path(args.dataset)), ensure_ascii=False), flush=True)

    import torch

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    config, model, train_samples, eval_samples = build(args, base)
    print(f"Eğitim örneği: {len(train_samples)}, doğrulama: {len(eval_samples)}", flush=True)

    if args.smoke:
        ck = smoke(config, model, train_samples)
        del model
        torch.cuda.empty_cache()
        export(ck, base, out, pick_reference(Path(args.dataset)))
        return

    from trainer import Trainer, TrainerArgs

    trainer = Trainer(
        TrainerArgs(restore_path=None, skip_train_epoch=False, start_with_eval=True,
                    grad_accum_steps=args.grad_accum, continue_path=args.continue_path or ""),
        config, output_path=config.output_path, model=model,
        train_samples=train_samples, eval_samples=eval_samples,
    )
    trainer.fit()
    best = find_best(Path(config.output_path))
    print(f"En iyi checkpoint: {best}", flush=True)
    export(best, base, out, pick_reference(Path(args.dataset)))


args_global = None

if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/eval_xtts.py
"""Fine-tune öncesi/sonrası karşılaştırma: aynı cümleler, aynı referans ses.

    python eval_xtts.py --models temel=/content/xtts_base ince=/content/drive/.../hazir_model \
        --ref .../hazir_model/reference.wav --out /content/drive/.../degerlendirme

Her model için: Whisper ile anlaşılırlık (WER, düşük iyi) ve ECAPA ile referansa ses benzerliği
(yüksek iyi). Dinlemek için wav'lar da yazılır; son karar kulakla verilmeli.
"""

from __future__ import annotations

import argparse
import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF

SENTENCES = [
    "Merhaba Yunus, bugün nasılsın?",
    "Ankara'da hava parçalı bulutlu, akşama doğru yağmur bekleniyor.",
    "Veri yapıları sınavın için endişelenme, birlikte tekrar edebiliriz.",
    "Saat on dört, otuz beş. Toplantına yarım saat kaldı.",
    "Bunu hiç düşünmemiştim, gerçekten ilginç bir fikir!",
    "Işıklar için henüz bir bağlantım yok ama yakında olacak.",
]


def wer(ref: str, hyp: str) -> float:
    norm = lambda s: re.sub(r"[^\w\s]", " ", s.replace("I", "ı").replace("İ", "i").lower()).split()
    r, h = norm(ref), norm(hyp)
    d = list(range(len(h) + 1))
    for i, rw in enumerate(r, 1):
        prev, d[0] = d[0], i
        for j, hw in enumerate(h, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (rw != hw))
    return d[-1] / max(1, len(r))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--models", nargs="+", required=True, help="ad=klasör (model.pth, config.json, vocab.json)")
    ap.add_argument("--ref", required=True, help="Klon referansı (hedef konuşmacının temiz klibi)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--whisper", default="large-v3")
    args = ap.parse_args()

    from speechbrain.inference.speaker import EncoderClassifier
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts

    enc = EncoderClassifier.from_hparams("speechbrain/spkrec-ecapa-voxceleb", savedir="/tmp/ecapa",
                                         run_opts={"device": "cuda"})

    def emb(wav, sr):
        e = enc.encode_batch(AF.resample(torch.as_tensor(wav, dtype=torch.float32), sr, 16000)[None].cuda())
        e = e.flatten().cpu().numpy()
        return e / np.linalg.norm(e)

    ref_wav, ref_sr = sf.read(args.ref, dtype="float32")
    ref_emb = emb(ref_wav if ref_wav.ndim == 1 else ref_wav.mean(1), ref_sr)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    results = {}
    audio = {}
    for spec in args.models:
        name, d = spec.split("=", 1)
        cfg = XttsConfig()
        cfg.load_json(str(Path(d) / "config.json"))
        m = Xtts.init_from_config(cfg)
        m.load_checkpoint(cfg, checkpoint_dir=d, eval=True)
        m.cuda()
        g, s = m.get_conditioning_latents(audio_path=[args.ref], gpt_cond_len=cfg.gpt_cond_len,
                                          gpt_cond_chunk_len=cfg.gpt_cond_chunk_len, max_ref_length=cfg.max_ref_len)
        audio[name] = []
        sims, t0 = [], time.perf_counter()
        for i, text in enumerate(SENTENCES):
            torch.manual_seed(i)
            w = np.asarray(m.inference(text, "tr", g, s, temperature=0.65)["wav"], dtype=np.float32)
            sf.write(out / f"{name}_{i}.wav", w, 24000)
            audio[name].append(w)
            sims.append(float(emb(w, 24000) @ ref_emb))
        results[name] = {"benzerlik": round(float(np.mean(sims)), 3), "sure_s": round(time.perf_counter() - t0, 1)}
        del m, g, s
        gc.collect()
        torch.cuda.empty_cache()
    del enc
    gc.collect()
    torch.cuda.empty_cache()

    from faster_whisper import WhisperModel

    wh = WhisperModel(args.whisper, device="cuda", compute_type="int8_float16")
    for name, wavs in audio.items():
        ws = []
        for text, w in zip(SENTENCES, wavs):
            segs, _ = wh.transcribe(AF.resample(torch.from_numpy(w), 24000, 16000).numpy(), language="tr",
                                    beam_size=5, condition_on_previous_text=False)
            ws.append(wer(text, " ".join(x.text for x in segs)))
        results[name]["wer"] = round(float(np.mean(ws)), 3)

    (out / "sonuc.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
    print("\n| model | referansa benzerlik (↑) | WER (↓) |\n|---|---|---|")
    for name, r in results.items():
        print(f"| {name} | {r['benzerlik']:.3f} | {r['wer']:.3f} |")
    print(f"\nSesler: {out}")


if __name__ == "__main__":
    main()

## 5. Eğitim
İlerlemeyi görmek için isteğe bağlı: `%load_ext tensorboard` ve `%tensorboard --logdir {OUT_DIR}/egitim`

In [ ]:
cont = f'--continue-path "{CONTINUE_PATH}"' if CONTINUE_PATH else ''
!cd /content && {PY} train_xtts.py --dataset {DATASET_DIR} --out {OUT_DIR} --epochs {EPOCHS} \
    --batch {BATCH} --grad-accum {GRAD_ACCUM} --lr {LR} --workers 8 {cont}

## 6. Değerlendirme: temel model ve fine-tune edilmiş model aynı cümleleri okur

In [ ]:
!cd /content && {PY} eval_xtts.py --models temel=/content/xtts_base ince={OUT_DIR}/hazir_model \
    --ref {OUT_DIR}/hazir_model/reference.wav --out {OUT_DIR}/degerlendirme
from IPython.display import Audio, display
for i in range(6):
    for name in ('temel', 'ince'):
        print(name, i); display(Audio(f'{OUT_DIR}/degerlendirme/{name}_{i}.wav'))

## 7. İndir ve TOM'da kullan

`OUT_DIR/hazir_model` klasörünü (≈1,9 GB) Drive'dan indir, sonra yerelde:

```bash
PYTHONPATH=. .venv/bin/python -m tom --deepspeed --xtts ~/indirilen/hazir_model --ref ~/indirilen/hazir_model/reference.wav
```

Daha uzun bir referans istersen `--ref` ile veri setinden birkaç temiz klip verebilirsin.

In [ ]:
import shutil
shutil.make_archive('/content/hazir_model', 'zip', f'{OUT_DIR}/hazir_model')
from google.colab import files
files.download('/content/hazir_model.zip')   # ya da Drive'dan indir